# Orchestrate: issue에서 merge된 PR까지

이 notebook은 agent가 현실적인 전체 루프를 수행하도록 안내합니다. 모호한 bug report 읽기, 버그 찾기, 수정하기, PR 열기, CI 통과하기, review feedback 처리하기, merge하기입니다. 실제 maintainer workflow는 결코 일직선이 아니며 이것이 연습의 핵심입니다. agent는 여러 tool 타입(JSON 읽기, 코드 grep, 파일 편집, 모의 CLI 실행, CI 출력 parsing)에 걸쳐 상태를 유지하면서 체인 도중 발생하는 두 가지 뜻밖의 상황, CI 실패와 docstring을 요구하는 review bot에서 복구해야 합니다.

상태는 issue 본문 → 파일 경로 → 수정 diff → PR 번호 → CI 출력 → review comment → 최종 merge로 이어집니다. fixture의 `gh-mock` CLI는 모든 것을 `.gh-state/`에 저장하므로 각 단계는 이전 단계에서 수행한 작업을 볼 수 있습니다. 네트워크 접근 없이 실제 GitHub workflow를 흉내 냅니다.

iterate notebook에 더하여 배울 내용은 다음과 같습니다.

- **긴 체인을 여러 turn에 걸쳐 이끌기.** 세션 filesystem과 대화 이력은 turn 사이에 유지되므로 각 user message는 이전 message가 끝난 지점에서 이어갑니다. 이를 마지막에 최종 상태를 검증하는 데 사용합니다.
- **체인 도중 복구하기.** agent는 무작정 재시도하지 않고 CI 실패나 review comment를 읽고 대응해야 합니다.

fixture는 `example_data/orchestrate/`에 있으며 모의 `gh` CLI, issue JSON 파일, 버그를 심어 둔 `src/` + `tests/` 구조가 들어 있습니다.


In [ ]:
import io
import os
import zipfile
from pathlib import Path

from anthropic import Anthropic
from utilities import stream_until_end_turn, wait_for_idle_status

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")
GH_TOKEN = os.environ.get("GITHUB_TOKEN")  # github_repository 보충 설명에만 필요합니다

client = Anthropic()
FIXTURE = Path("example_data") / "orchestrate"

## 1. fixture 묶기

모의 repository는 몇 개의 파일로 구성됩니다. `src/url_utils.py`에 실제 버그가 있고, `src/blog.py`는 버그를 관찰하기 쉽게 하는 호출자이며, `tests/test_urls.py`는 버그를 수정하기 전까지 실패합니다. `gh-mock` CLI와 `issue_42.json`은 agent가 진행할 GitHub와 유사한 workflow를 제공합니다. 디렉터리를 메모리에서 zip으로 압축하고 단일 파일 resource로 업로드합니다. 대신 실제 GitHub repository를 mount하는 방법은 마지막 보충 설명을 참고하세요.


In [ ]:
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as zf:
    for f in FIXTURE.rglob("*"):
        if f.is_file() and f.name != "README.md":
            zf.write(f, f.relative_to(FIXTURE))
buf.seek(0)
fixture_zip = client.beta.files.upload(file=("repo.zip", buf, "application/zip"))
print(f"fixture: {fixture_zip.id}")

## 2. Agent + environment + session

environment는 `pytest`를 pip dependency로 선언하므로 agent는 CI 루프의 일부로 실제 테스트 suite를 실행할 수 있습니다. package 설치에 네트워크 접근이 필요한 cookbook의 첫 notebook이므로, 나머지는 `limited`인 networking 설정에 `allow_package_managers: True`가 들어 있습니다.


In [ ]:
agent = client.beta.agents.create(
    name="cookbook-orchestrate",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-orchestrate-issue-to-pr"},
    model=MODEL,
    system=(
        "You are a maintainer bot. You read issues via `./gh-mock`, explore "
        "the codebase, write fixes, and shepherd PRs through CI and review. "
        "When CI fails or a reviewer requests changes, read what they said "
        "and address it, don't just retry blindly.\n\n"
        "Work in /mnt/user. All gh-mock commands run from there."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

env = client.beta.environments.create(
    name="cookbook-orchestrate-env",
    config={
        "type": "cloud",
        "networking": {"type": "limited", "allow_package_managers": True},
        "packages": {"pip": ["pytest"]},
    },
)

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[{"type": "file", "file_id": fixture_zip.id, "mount_path": "repo.zip"}],
    title="Issue #42 → PR",
)
print(f"session: {session.id}")

## 3. 전체 체인 실행하기

하나의 지시로 전체 루프를 시작합니다. 살펴볼 두 복구 지점은 CI 실패와 review comment입니다. agent의 첫 수정이 불완전하면, 예를 들어 `é`만 처리하고 `ü`를 놓치면 `gh-mock pr checks`는 0이 아닌 종료 코드와 agent가 읽고 반복해야 하는 pytest 출력을 반환합니다. CI가 통과한 뒤에는 `slugify()`에 docstring이 없으면 reviewer bot이 merge를 막아, 최종 merge 전에 agent가 한 번 더 대응할 기회를 줍니다.


In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Unpack /mnt/session/uploads/repo.zip into /mnt/user "
                        "and ship a fix for issue #42 end-to-end. Read the "
                        "./gh-mock script first to see what subcommands it "
                        "supports; use those to view the issue, open a PR, "
                        "run CI, handle review feedback, and merge. Show me "
                        "the final PR state when you're done."
                    ),
                }
            ],
        }
    ],
)

print("=== full orchestrate chain ===")
stream_until_end_turn(client, session.id)

## 4. Multi-turn 검증

세션은 stateful입니다. container filesystem과 대화 이력이 turn 사이에 유지되므로 후속 작업은 또 다른 `user.message`를 보내기만 하면 됩니다. 여기서는 최종 상태를 독립적으로 검증하는 데 사용합니다. 모의 CLI가 `.gh-state/pr_101.json`에 PR 상태를 저장하므로 해당 파일을 출력하는 것이 CI가 통과하고 merge 전에 적어도 하나의 review 승인을 받은 뒤 PR이 merge되었는지 확인하는 가장 간단한 방법입니다.


In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Print the contents of /mnt/user/.gh-state/pr_101.json so "
                        "I can see the final state, CI status, and reviews."
                    ),
                }
            ],
        }
    ],
)
stream_until_end_turn(client, session.id)

wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
print("archived")

## 보충 설명: 실제 GitHub repository mount하기

위 fixture는 notebook을 offline으로 실행하고 GitHub credential 없이 사용해 볼 수 있도록 모의 환경입니다. 실제 repository에서 실제 작업을 하려면 위의 `{"type": "file", ...}` mount를 `{"type": "github_repository", ...}` resource로 바꾸세요. API가 세션 시작 시 container에 repo를 clone합니다.

iterate notebook의 파일 mount와 같은 `resources=` 필드입니다. 목록은 여러 타입을 혼합하여 받을 수 있으므로 같은 호출에서 repo를 clone하면서 별도 config 파일도 mount할 수 있습니다. agent의 bash/read/grep tool은 working tree를 일반 디렉터리로 봅니다. 유일한 차이는 직접 clone하는 대신 API가 처리한다는 것입니다.

```python
session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[
        {
            "type": "github_repository",
            "url": "https://github.com/anthropics/claude-cookbooks",
            "mount_path": "/workspace/cookbook",
            "authorization_token": GH_TOKEN,
            "checkout": {"type": "branch", "name": "main"},
        }
    ],
    title="Repo explorer",
)
```

private repo와 clone 인증에는 `GITHUB_TOKEN`이 필요합니다. clone은 세션 생성 시 한 번 이루어지며, 이후 turn은 다시 clone하지 않고 같은 working tree에서 작업합니다.

repository mount는 한 가지 일을 더 합니다. repo 루트에 `.claude/skills/` 디렉터리가 있으면 세션 시작 시 그 안의 모든 skill을 발견하여 agent에 주입합니다. `skills` 필드는 필요 없습니다. 자체 build, test, review skill을 제공하는 repo에서 작업하는 code agent는 사람 기여자와 같은 방식으로 이를 받아들입니다. [`CMA_use_skills_from_a_repo.ipynb`](CMA_use_skills_from_a_repo.ipynb)에서는 구조와 먼저 읽고 따르는 protocol을 설명합니다.
